# Real-data grid: results

Reads the per-episode dumps in `results/real/<dataset>/<cell>_<tag>_o<offset>.json` (the shards of each cell),
and reports:

1. **Inventory**: which cells are complete.
2. **Within-run contrasts**: the copula row (`icl`) minus independence, the autoregressive row and best_baseline,
   with 95 % cluster-bootstrap CIs (2,000 replicates) and win rates (`eval.runners.paired_compare`).
3. **Calibration gate**: the marginal's PIT `z_test` against N(0, 1); std outside [0.8, 1.25] is flagged
   (`eval.runners.real_calibration_gate`).
4. **Mean total NLL per method**.
5. **CTR23**: the Wilcoxon signed-rank test across datasets (the dataset is the unit).
6. **Cross-cell paired comparisons** (e.g. California f1 − f0 on identical episodes).
7. **Plots**.
8. **Context size**: the contrasts and method NLLs against $P$ for the cells swept over it.
9. A **per-episode DataFrame** for your own analysis.

**Units and signs.** Every number is a per-point NLL in nats on the raw $y$ scale:
$\text{NLL} = -\tfrac{1}{n}\sum_i \log p(y_i) = \text{marginal} + \text{copula}$.
A delta is $\Delta_e = \text{NLL}_{\text{copula}}(e) - \text{NLL}_{\text{ref}}(e)$ per episode $e$;
**negative = the copula is better**; win rate $= \Pr_e[\Delta_e < 0]$.
Meuse folds share points, so it has no valid CI: it shows mean ± sd.

`best_gp_total` in the method table is the best GP *per episode*, chosen on the test score: it is an
oracle-selected lower bound, not a fair baseline. The fair comparison is `best_baseline` (cross-fitted).

## How to run

Run it where the dumps and the cluster venv are, i.e. on a Grid5000 node, not on the laptop:

- **VS Code Remote-SSH** to a reserved node, open this file, pick the main checkout's `.venv` as kernel,
  Run All. The setup cell puts *this worktree* first on `sys.path`, so the worktree's runners are used.
- **Headless** (from the worktree root, inside an OAR job):
  ```bash
  source scripts/_env.sh && uv run --active --no-sync --with nbconvert --with ipykernel jupyter nbconvert \
      --to notebook --execute --inplace notebooks/real_results.ipynb --ExecutePreprocessor.kernel_name=python3
  ```
  Keep `--active`: without it, `uv run` in a worktree creates an empty `.venv` there, which `_env.sh`
  would then pick up for every later job (delete it if it appears).
  then open the saved notebook in VS Code to read the outputs.

Everything is light (JSON reading + bootstrap): a few minutes on one CPU node.

In [ ]:
# --- setup ---
import functools
import glob
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval/runners/paired_compare.py").exists())
for p in (ROOT / "tabicl_upstream/src", ROOT, ROOT / "src"):  # ends as src, ROOT, tabicl first
    if str(p) in sys.path:
        sys.path.remove(str(p))
    sys.path.insert(0, str(p))

from eval.runners.paired_compare import (  # noqa: E402
    PairedCompareSpec,
    across_datasets,
    paired_deltas,
    within_run_deltas,
)
from eval.runners.real_calibration_gate import gate_stats  # noqa: E402

RESULTS = ROOT / "results/real"
TAG = "nano"  # REAL_CKPT_TAG of the checkpoint pair
N_BOOT = 2000
EXPECTED = {"meuse": 201, "ctr23": 100}  # episodes per cell; 400 otherwise
pd.set_option("display.max_rows", 200, "display.width", 200, "display.max_colwidth", 80)
print("worktree:", ROOT)

## 1. Inventory

In [ ]:
@functools.lru_cache(maxsize=None)
def load_eps(cell: str) -> tuple:
    # Episodes of every shard of a cell, sorted by ep_i.
    eps = []
    for f in sorted(glob.glob(GLOB[cell])):
        with open(f) as fh:
            eps += json.load(fh)["episodes"]
    return tuple(sorted(eps, key=lambda e: int(e["ep_i"])))


# CTR23 cells are named by OpenML data_id (ctr23_44957_dyadic_knn); show the dataset name instead
# (ctr23_airfoil_self_noise_dyadic_knn), taken from the suite audit (python -m eval.runners.real_audit).
_audit = RESULTS / "audit/ctr23_suite.json"
CTR23_NAMES = (
    {int(k): v["name"] for k, v in json.loads(_audit.read_text())["suite"]["datasets"].items()}
    if _audit.is_file()
    else {}
)


def cell_label(cell: str) -> str:
    # ctr23_<data_id>_<mask> -> ctr23_<name>_<mask>; every other cell keeps its file name.
    m = re.match(r"^ctr23_(\d+)_(.+)$", cell)
    return f"ctr23_{CTR23_NAMES.get(int(m[1]), m[1])}_{m[2]}" if m else cell


pat = re.compile(rf"^(?P<cell>.+)_{TAG}_o(?P<off>\d+)\.json$")
rows = []
for f in sorted(RESULTS.glob(f"*/*_{TAG}_o*.json")):
    m = pat.match(f.name)
    if m:
        rows.append(
            {"dataset": f.parent.name, "file_cell": m["cell"], "cell": cell_label(m["cell"]), "offset": int(m["off"])}
        )
shards = pd.DataFrame(rows)
# o*[0-9].json: the shards only, not their *_results_partial.json resume files.
GLOB = {r.cell: str(RESULTS / r.dataset / f"{r.file_cell}_{TAG}_o*[0-9].json") for r in shards.itertuples()}
inv = shards.groupby(["dataset", "cell"]).agg(shards=("offset", "size")).reset_index()
inv["episodes"] = [len(load_eps(c)) for c in inv.cell]
inv["expected"] = inv.dataset.map(EXPECTED).fillna(400).astype(int)
inv["complete"] = inv.episodes >= inv.expected
COMPLETE = list(inv.loc[inv.complete, "cell"])
print(f"{len(inv)} cells, {inv.complete.sum()} complete, {inv.shards.sum()} shards")
inv[~inv.complete] if (~inv.complete).any() else print("all cells complete")

In [ ]:
inv.groupby("dataset").agg(cells=("cell", "size"), complete=("complete", "sum"), episodes=("episodes", "sum"))

## 2. Within-run contrasts (copula − reference)

Headline term: `total` for independence and the autoregressive row (same marginal, so this is also the copula
gain); `copula (shared z)` for best_baseline, which is scored in z-space only.
Set `INCLUDE_INCOMPLETE = True` to also see cells still running.

In [ ]:
INCLUDE_INCOMPLETE = False


def _spec(**kw) -> PairedCompareSpec:
    return PairedCompareSpec(n_boot=N_BOOT, **kw)


def _rows(res: dict, **ids) -> list[dict]:
    return [dict(**ids, contrast=c, term=t, **s) for c, terms in res.items() for t, s in terms.items() if s.get("n")]


def within(cell: str, folds: tuple = ()) -> pd.DataFrame:
    # Copula minus independence / autoregressive / best_baseline in one cell (all NLL terms).
    return pd.DataFrame(_rows(within_run_deltas(_spec(a=GLOB[cell], folds=list(folds))), cell=cell))


def compare(cell_a: str, cell_b: str) -> pd.DataFrame:
    # B - A per method on the episodes both cells share (raises if the episode meta differs).
    return pd.DataFrame(_rows(paired_deltas(_spec(a=GLOB[cell_a], b=GLOB[cell_b])), a=cell_a, b=cell_b))


def fmt(r) -> str:
    ci = f"[{r.lo:+.3f}, {r.hi:+.3f}]" if np.isfinite(r.lo) else f"± {r.sd:.3f} sd"
    return f"{r.mean:+.3f} {ci}  win {r.win_rate:.2f}"


cells_w = [c for c in inv.cell if INCLUDE_INCOMPLETE or c in COMPLETE]
W = pd.concat([within(c) for c in cells_w], ignore_index=True)
W["dataset"] = W.cell.map(dict(zip(inv.cell, inv.dataset)))
HEAD = W[W.term.isin(["total", "copula (shared z)"])].copy()
HEAD["cell_txt"] = [fmt(r) for r in HEAD.itertuples()]
headline = HEAD.pivot(index="cell", columns="contrast", values="cell_txt").reindex(cells_w)
headline[~headline.index.str.startswith("ctr23_")]

In [ ]:
# The same with the significance call: the CI excludes 0 (or |mean| > sd for Meuse, as a rough read only).
def verdict(r) -> str:
    if np.isfinite(r.lo):
        return "copula better" if r.hi < 0 else ("copula worse" if r.lo > 0 else "n.s.")
    return "(no CI)"


HEAD["verdict"] = [verdict(r) for r in HEAD.itertuples()]
HEAD.pivot(index="cell", columns="contrast", values="verdict").reindex(cells_w).loc[
    lambda d: ~d.index.str.startswith("ctr23_")
]

In [ ]:
# Full table, every term (total / marginal / copula), for one dataset:
DATASET = "kaust"
W[W.dataset == DATASET][["cell", "contrast", "term", "n", "n_clusters", "mean", "lo", "hi", "sd", "win_rate"]]

## 3. Calibration gate (marginal PIT `z_test`)

In [ ]:
def gate(cell: str) -> dict:
    z = np.array([v for e in load_eps(cell) for v in e.get("z_test", [])], dtype=float)
    return {"cell": cell, **gate_stats(z)}


G = pd.DataFrame([gate(c) for c in cells_w]).set_index("cell")
print(f"flagged (std outside [0.8, 1.25]): {list(G.index[G.flag])}")
G[["n", "mean", "std", "frac_abs_gt3", "ks_stat", "quantile_ece", "flag"]].round(4)

## 4. Mean total NLL per method

Per-point nats, averaged over episodes. EMNIST in bits/pixel is $(\text{NLL} + \ln 256)/\ln 2$.

In [ ]:
SKIP = {"oracle_prior", "oracle_posterior"}


def method_means(cell: str) -> pd.Series:
    eps = load_eps(cell)
    keys = sorted(set().union(*(e["total_nlls"] for e in eps)) - SKIP)
    return pd.Series(
        {k: np.nanmean([float((e["total_nlls"].get(k) or {}).get("total", np.nan)) for e in eps]) for k in keys},
        name=cell,
    )


M = pd.concat([method_means(c) for c in cells_w if not c.startswith("ctr23_")], axis=1)
order = ["icl", "autoregressive", "independence_marginal"]
M = M.reindex(order + [k for k in M.index if k not in order])
M.round(3)

In [ ]:
# Rank of the copula among methods with a real total (best_gp_total excluded: oracle-selected).
R = M.drop(index=["best_gp_total"], errors="ignore").rank(axis=0)
R.loc[["icl", "autoregressive", "independence_marginal"]].astype("Int64")

## 5. CTR23: across-dataset Wilcoxon test

Per dataset, the mean within-run delta; then the Wilcoxon signed-rank test of those per-dataset means
against 0. `better` = datasets where the copula's mean delta is negative. Only complete cells enter.

In [ ]:
def ctr23_across(mask: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    cs = [c for c in COMPLETE if c.startswith("ctr23_") and c.endswith(f"_{mask}")]
    res = across_datasets(_spec(across=[GLOB[c] for c in cs]))
    summary = pd.DataFrame(
        [
            {
                "contrast / term": k,
                **{x: v.get(x) for x in ("n_datasets", "n_better", "median", "mean", "wilcoxon_stat", "p_value")},
            }
            for k, v in res.items()
        ]
    ).set_index("contrast / term")
    per = pd.DataFrame({k: v["per_dataset"] for k, v in res.items()})
    return summary, per


CTR = {mask: ctr23_across(mask) for mask in ("dyadic_knn", "random")}
for mask, (summary, _) in CTR.items():
    print(f"\n=== mask {mask} ===")
    display(summary.round(4))

In [ ]:
# Per-dataset mean deltas (rows: datasets), mask dyadic_knn:
CTR["dyadic_knn"][1].round(4)

## 6. Cross-cell paired comparisons (B − A, identical episodes)

Each row is a method's NLL in cell B minus in cell A; negative = B better. A pair whose episodes differ
(different meta) raises and is skipped. P-sweep cells draw different episodes and are not paired.

In [ ]:
PAIRS = [
    ("california_f0_random", "california_f1_random"),
    ("california_f0_block", "california_f1_block"),
    ("meuse_f0", "meuse_f1"),
]
out = []
for a, b in PAIRS:
    if a not in GLOB or b not in GLOB:
        print(f"skip {b} - {a}: missing cell")
        continue
    if not (a in COMPLETE and b in COMPLETE):
        print(f"note {b} - {a}: a cell is incomplete, only shared episodes are compared")
    try:
        out.append(compare(a, b))
    except ValueError as err:
        print(f"skip {b} - {a}: {err}")
if out:
    P = pd.concat(out, ignore_index=True)
    P = P[P.contrast.isin(["icl", "autoregressive", "independence_marginal", "best_baseline"])]
    display(
        P[["a", "b", "contrast", "term", "n", "mean", "lo", "hi", "sd", "win_rate"]]
        .rename(columns={"contrast": "method"})
        .round(4)
    )

## 7. Plots

In [ ]:
# Forest plot: copula minus independence and minus autoregressive (total, per point), non-CTR23 cells.
fig, axes = plt.subplots(1, 2, figsize=(13, 0.32 * len(headline.index) + 1.5), sharey=True)
H = HEAD[~HEAD.cell.str.startswith("ctr23_")]
ylab = [c for c in cells_w if not c.startswith("ctr23_")]
for ax, contrast in zip(axes, ["copula - independence", "copula - autoregressive"]):
    d = H[(H.contrast == contrast) & (H.term == "total")].set_index("cell").reindex(ylab)
    y = np.arange(len(d))
    lo = np.where(np.isfinite(d.lo), d["mean"] - d.lo, d.sd)
    hi = np.where(np.isfinite(d.hi), d.hi - d["mean"], d.sd)
    ax.errorbar(d["mean"], y, xerr=[lo, hi], fmt="o", ms=4, capsize=2)
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_title(f"{contrast} (total NLL, nats/point)")
    ax.set_xlabel("Δ  (< 0: copula better)")
axes[0].set_yticks(np.arange(len(ylab)), ylab)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Per-episode distribution of Δ = copula - reference for one cell.
SHOW_CELL = "kaust_1a_2_random"
eps = load_eps(SHOW_CELL)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, ref in zip(axes, ["independence_marginal", "autoregressive"]):
    d = np.array([e["total_nlls"]["icl"]["total"] - e["total_nlls"][ref]["total"] for e in eps])
    ax.hist(d, bins=40)
    ax.axvline(0, color="k", lw=0.8)
    ax.axvline(np.mean(d), color="C1", lw=1.5, label=f"mean {np.mean(d):+.3f}, win {np.mean(d < 0):.2f}")
    ax.set_title(f"{SHOW_CELL}: copula − {ref}")
    ax.set_xlabel("Δ total NLL (nats/point)")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CTR23: per-dataset mean Δ (copula - independence, total), both masks.
fig, ax = plt.subplots(figsize=(12, 3.8))
key = "copula - independence / total"
per = pd.DataFrame({m: CTR[m][1].get(key) for m in CTR})
per.sort_values("dyadic_knn").plot.bar(ax=ax)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("mean Δ (nats/point)")
ax.set_title("CTR23: copula − independence per dataset (< 0: copula better)")
plt.tight_layout()
plt.show()

## 8. Context size (P sweep)

A *family* is a cell name with its `_p<P>` token removed (e.g. `kaust_1a_2_random{,_p64,_p256}`,
`emnist_p{50,200}_block`); only families with at least two context sizes are shown. $P$ is read from the
episode meta (`episode["P"]`), and the number of test points $N = 64$ is the same everywhere.

- **Top row**: the within-run contrasts of section 2, $\bar\Delta(P) = \overline{\text{NLL}_{\text{copula}} - \text{NLL}_{\text{ref}}}$
  (total, nats/point) with their 95 % cluster-bootstrap CIs. Below 0, the copula is better.
- **Bottom row**: mean total NLL of each method $\pm$ 1 s.e. over episodes.

Each $P$ draws its own episodes, so a trend across $P$ compares independent samples (not paired); only the
within-$P$ CIs are paired.

In [ ]:
def family(cell: str) -> str:
    # Cell name without its context-size token: kaust_1a_2_random_p64 -> kaust_1a_2_random.
    return re.sub(r"_p\d+(?=_|$)", "", cell)


def context_size(cell: str) -> int:
    # Most common episode P of the cell (Meuse folds differ by one point).
    return int(pd.Series([int((e.get("episode") or {}).get("P", -1)) for e in load_eps(cell)]).mode().iloc[0])


def method_stats(cell: str, method: str) -> tuple[float, float]:
    # Mean total NLL of a method over episodes and its standard error.
    v = np.array([float((e["total_nlls"].get(method) or {}).get("total", np.nan)) for e in load_eps(cell)])
    v = v[np.isfinite(v)]
    return (v.mean(), v.std(ddof=1) / np.sqrt(v.size)) if v.size > 1 else (np.nan, np.nan)


SWEEP_METHODS = ["icl", "autoregressive", "independence_marginal"]
SWEEP_CONTRASTS = ["copula - independence", "copula - autoregressive"]

S = pd.DataFrame({"cell": [c for c in cells_w if not c.startswith("ctr23_")]})
S["dataset"] = S.cell.map(dict(zip(inv.cell, inv.dataset)))
S["family"] = S.cell.map(family)
S["P"] = S.cell.map(context_size)
S = S[S.groupby("family").P.transform("nunique") > 1].sort_values(["family", "P"]).reset_index(drop=True)
for m in SWEEP_METHODS:
    S[[f"{m}.mean", f"{m}.se"]] = [method_stats(c, m) for c in S.cell]
HT = HEAD[HEAD.term == "total"].set_index(["cell", "contrast"])
for con in SWEEP_CONTRASTS:
    for k in ("mean", "lo", "hi", "win_rate"):
        S[f"{con}.{k}"] = [HT[k].get((c, con), np.nan) for c in S.cell]

table = S.set_index(["family", "P"])
table_txt = pd.DataFrame(
    {
        con: [
            f"{r[f'{con}.mean']:+.3f} [{r[f'{con}.lo']:+.3f}, {r[f'{con}.hi']:+.3f}]  win {r[f'{con}.win_rate']:.2f}"
            for _, r in table.iterrows()
        ]
        for con in SWEEP_CONTRASTS
    }
    | {f"{m} NLL": [f"{r[f'{m}.mean']:.3f} ± {r[f'{m}.se']:.3f}" for _, r in table.iterrows()] for m in SWEEP_METHODS},
    index=table.index,
)
print(f"{S.family.nunique()} families with a context-size sweep: {sorted(S.family.unique())}")
table_txt

In [ ]:
fams = list(dict.fromkeys(S.family))
if not fams:
    print("no family with more than one context size (set INCLUDE_INCOMPLETE = True if they are still running)")
else:
    fig, axes = plt.subplots(2, len(fams), figsize=(3.6 * len(fams), 6.5), squeeze=False, sharex="col")
    for j, fam in enumerate(fams):
        d = S[S.family == fam]
        ax = axes[0, j]
        for k, con in enumerate(SWEEP_CONTRASTS):
            m, lo, hi = d[f"{con}.mean"], d[f"{con}.lo"], d[f"{con}.hi"]
            x = d.P * (1 + 0.03 * (k - 0.5))  # small horizontal dodge so the CIs do not overlap
            ax.errorbar(x, m, yerr=[m - lo, hi - m], fmt="o-", ms=4, capsize=2, label=con.replace("copula - ", "− "))
        ax.axhline(0, color="grey", lw=0.8)
        ax.set_title(fam, fontsize=9)
        ax = axes[1, j]
        for m in SWEEP_METHODS:
            ax.errorbar(d.P, d[f"{m}.mean"], yerr=d[f"{m}.se"], fmt="o-", ms=4, capsize=2, label=m)
        ax.set_xscale("log", base=2)
        ax.set_xticks(d.P, [str(p) for p in d.P])
        ax.minorticks_off()
        ax.set_xlabel("context size P")
    axes[0, 0].set_ylabel("Δ total NLL (nats/point)\n< 0: copula better")
    axes[1, 0].set_ylabel("mean total NLL (nats/point)")
    axes[0, 0].legend(fontsize=8)
    axes[1, 0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 9. Per-episode DataFrame for your own analysis

One row per episode: its meta (cluster, fold, …), `n_test`, and the total / marginal / copula NLL of each method.

In [ ]:
def episodes_df(cell: str) -> pd.DataFrame:
    rows = []
    for e in load_eps(cell):
        meta = {k: v for k, v in (e.get("episode") or {}).items() if not isinstance(v, (list, dict))}
        r = {"ep_i": e["ep_i"], "n_test": e.get("n_test"), **meta}
        for m, terms in e["total_nlls"].items():
            for t, v in (terms or {}).items():
                r[f"{m}.{t}"] = v
        rows.append(r)
    return pd.DataFrame(rows).set_index("ep_i")


df = episodes_df(SHOW_CELL)
df[["cluster", "fold", "n_test", "icl.total", "autoregressive.total", "independence_marginal.total"]].head()

## 9. CTR23 on the full datasets: copula vs independence

`eval.runners.ctr23_full_eval` (docs/workflows.md, "OpenML-CTR23 on the full datasets"): the task's official
10-fold CV, the whole training fold as context and the whole test fold as queries, one JSON per dataset in
`results/real/ctr23_full/`. Independence is the marginal alone (copula NLL 0), so per point

$$\Delta_T = \text{NLL}_\text{total}(\text{copula}) - \text{NLL}_\text{total}(\text{independence}) = \text{copula NLL over } T,$$

for $T$ = the whole test fold, or its partition into 64-row chunks (`random`, `dyadic_knn`). Negative = the copula
is better. Per dataset: mean over folds and the fold win rate; across datasets: mean / median $\Delta$, wins,
mean and **median rank** of the two methods (1 = better), and the Wilcoxon signed-rank test of the per-dataset
$\Delta$ (the dataset is the unit). With two methods a rank is 1 or 2, so mean rank = 1 + loss share.

In [ ]:
from eval.runners.ctr23_full_eval import TARGETS, compare_with_independence

FULL_DIR = RESULTS / "ctr23_full"
INCLUDE_PARTIAL = False  # True: also datasets whose 10 folds are not all done
full_blobs = []
# Exact method match: "*_nano.json" alone would also match "*_kumo_ft_nano.json".
for f in sorted(p for p in FULL_DIR.glob(f"*_{TAG}.json") if p.stem.split("_", 1)[1] == TAG):
    with open(f) as fh:
        full_blobs.append(json.load(fh))
print(
    f"{len(full_blobs)} datasets with results; folds done:", {b["dataset"]["name"]: len(b["folds"]) for b in full_blobs}
)
CMP = compare_with_independence(full_blobs, include_partial=INCLUDE_PARTIAL)
pd.DataFrame(CMP["summary"]).T.round(4)

In [ ]:
# Per dataset: copula - independence (nats/point) and fold win rate, per query set.
FD = pd.DataFrame(CMP["per_dataset"]).set_index("name").sort_values("n_rows")
cols = ["n_rows", "n_folds", "marginal_nll"] + [f"{p}_{t}" for t in TARGETS for p in ("delta", "fold_win_rate")]
FD[cols].round(4)

In [ ]:
# Marginal metrics (raw target units, mean over folds): comparable with published CTR23 results
# on the same official splits (e.g. RMSE / R^2 tables).
FD[["n_rows", "rmse", "r2", "mae_median", "crps", "marginal_nll", "z_test_std", "mean_abs_offdiag_corr"]].round(4)

In [ ]:
# Copula - independence per dataset, ordered by size: does the gain hold as P grows?
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(FD))
for j, t in enumerate(TARGETS):
    ax.bar(x + (j - 1) * 0.27, FD[f"delta_{t}"], width=0.27, label=t)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x, [f"{n}\n(n={r})" for n, r in zip(FD.index, FD.n_rows)], rotation=70, fontsize=7)
ax.set_ylabel("Δ total NLL (nats/point, < 0: copula better)")
ax.set_title("CTR23 full datasets: copula − independence")
ax.legend()
plt.tight_layout()
plt.show()

## 10. CTR23 full datasets: ranking against other tabular foundation models

Same official folds and full training-fold context for every method. Three copula rows: a nano copula on the
ERA5 fine-tune of TabICL, Kumo small or TabPFN-3.5 Fast, each with the marginal it was trained on (the runner
checks the pairing). Every other row is a marginal alone (the independence model, copula NLL 0): those three
fine-tuned marginals (the marginal half of the copula runs) and the pretrained TabICL v2, Kumo and TabPFN.
The TabICL copula is rank 512 trained 630k steps; the Kumo and TabPFN copulas are rank 128 trained 100k steps.
Ranks are per dataset (1 = best, ties averaged), over the datasets **every** method completed; a run that ran
out of memory on a 48 GB GPU leaves its dataset out (listed as excluded). Reported: mean and **median rank**,
datasets ranked first, the Friedman test, and pairwise Wilcoxon signed-rank tests with Holm correction.

- Joint NLL (`NLL_TARGET`): the copula's total NLL over the whole test fold (or a 64-row partition); a marginal's
  is its marginal NLL.
- CRPS and RMSE depend on the marginal only, so they rank the marginals (the copula row equals its marginal).

In [ ]:
from eval.runners.ctr23_full_eval import compare_methods, method_scores

NLL_TARGET = "whole"  # "whole", "dyadic_knn" or "random"


def load_runs(label: str) -> list:
    blobs = []
    # Exact method match: "*_nano.json" alone would also match "*_kumo_ft_nano.json".
    for f in sorted(p for p in FULL_DIR.glob(f"*_{label}.json") if p.stem.split("_", 1)[1] == label):
        with open(f) as fh:
            blobs.append(json.load(fh))
    return blobs


RUNS = {
    label: load_runs(label)
    for label in (TAG, "kumo_ft_nano", "tabpfn_ft_nano", "tabicl_base", "tabpfn35_fast", "kumo_small")
    + ("pre_tabicl_nano", "pre_tabpfn_nano", "pre_kumo_nano")
}
print({k: f"{sum(len(b['folds']) == b['dataset']['n_folds'] and not b.get('failed') for b in v)} complete"
       + (f", failed: {[b['dataset']['name'] for b in v if b.get('failed')]}" if any(b.get('failed') for b in v) else "")
       for k, v in RUNS.items()})
# A copula run's marginal alone is its marginal's independence row (same test forward). Each copula is scored
# with the marginal it was trained on: ERA5 fine-tuned (FT) or pretrained (the PRE_nano_PN_* copulas).
COPULAS = {
    "TabICL-FT + copula": RUNS[TAG],
    "Kumo-FT + copula": RUNS["kumo_ft_nano"],
    "TabPFN-FT + copula": RUNS["tabpfn_ft_nano"],
    "TabICL v2 base + copula": RUNS["pre_tabicl_nano"],
    "TabPFN-3.5 Fast + copula": RUNS["pre_tabpfn_nano"],
    "Kumo small + copula": RUNS["pre_kumo_nano"],
}
MARGINALS = {
    "TabICL-FT": RUNS[TAG],
    "Kumo-FT": RUNS["kumo_ft_nano"],
    "TabPFN-FT": RUNS["tabpfn_ft_nano"],
    "TabICL v2 base": RUNS["tabicl_base"],
    "TabPFN-3.5 Fast": RUNS["tabpfn35_fast"],
    "Kumo small": RUNS["kumo_small"],
}


def ranking(scores: dict) -> tuple:
    out = compare_methods({k: v for k, v in scores.items() if v})
    summ = pd.DataFrame(out["summary"]).T.sort_values("mean_rank")
    pair = pd.DataFrame(out["pairwise"])
    print(f"{len(out['datasets'])} datasets ranked; excluded: {out['excluded']}")
    if "friedman" in out:
        print(f"Friedman chi2 = {out['friedman']['stat']:.2f}, p = {out['friedman']['p_value']:.3g}")
    return out, summ, pair


nll_scores = {f"{name} ({NLL_TARGET})": method_scores(blobs, f"total_{NLL_TARGET}") for name, blobs in COPULAS.items()}
nll_scores |= {name: method_scores(blobs, "marginal_nll") for name, blobs in MARGINALS.items()}
NLL_OUT, nll_summary, nll_pairs = ranking(nll_scores)
nll_summary.round(3)

In [ ]:
nll_pairs.round(4)

In [ ]:
# Per-dataset ranks (rows: datasets), joint NLL.
pd.DataFrame(NLL_OUT["ranks"], index=NLL_OUT["datasets"], columns=NLL_OUT["methods"])

In [ ]:
# The marginals on CRPS and RMSE (per-dataset ranks are scale-free).
for metric in ("crps", "rmse"):
    print(f"\n=== {metric} ===")
    _, summ, pair = ranking({name: method_scores(blobs, metric) for name, blobs in MARGINALS.items()})
    display(summ.round(3))
    display(pair.round(4))

In [ ]:
# Mean rank with the median marked, joint NLL.
fig, ax = plt.subplots(figsize=(8, 0.5 * len(nll_summary) + 1))
y = np.arange(len(nll_summary))
ax.barh(y, nll_summary["mean_rank"], color="C0", alpha=0.6, label="mean rank")
ax.plot(nll_summary["median_rank"], y, "k|", ms=18, mew=2, label="median rank")
ax.set_yticks(y, nll_summary.index)
ax.invert_yaxis()
ax.set_xlabel("rank (1 = best)")
ax.set_title(f"CTR23 full datasets, joint NLL ({len(NLL_OUT['datasets'])} datasets)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()